In [13]:
import numpy as np
import random
import time

# ================= CONFIG =================
SHOW_TRACE = False  # Change to True if you want reasoning steps

# ================= SYMBOLIC SOLVER =================
class SymbolicSolver:
    def __init__(self):
        self.backtracks = 0
        self.constraint_checks = 0
        self.trace = []

    def is_valid(self, board, row, col, num):
        self.constraint_checks += 1
        if num in board[row]:
            return False
        if num in board[:, col]:
            return False
        br, bc = (row // 3) * 3, (col // 3) * 3
        if num in board[br:br+3, bc:bc+3]:
            return False
        return True

    def solve_pure(self, board):
        for i in range(9):
            for j in range(9):
                if board[i][j] == 0:
                    for num in range(1, 10):
                        if self.is_valid(board, i, j, num):
                            board[i][j] = num
                            if self.solve_pure(board):
                                return True
                            board[i][j] = 0
                            self.backtracks += 1
                    return False
        return True

# ================= NEURAL GUIDE (SIMULATED) =================
class NeuralGuide:
    def predict_cell_priority(self, board):
        priority = []
        for i in range(9):
            for j in range(9):
                if board[i][j] == 0:
                    options = sum(
                        1 for n in range(1, 10)
                        if self.quick_check(board, i, j, n)
                    )
                    priority.append((options, i, j))
        priority.sort()
        return [(i, j) for _, i, j in priority]

    def predict_values(self, board, row, col):
        values = []
        for num in range(1, 10):
            if self.quick_check(board, row, col, num):
                freq = np.count_nonzero(board == num)
                values.append((freq, num))
        values.sort()
        return [num for _, num in values]

    def quick_check(self, board, r, c, n):
        return (
            n not in board[r] and
            n not in board[:, c] and
            n not in board[3*(r//3):3*(r//3)+3, 3*(c//3):3*(c//3)+3]
        )

# ================= HYBRID SOLVER =================
class HybridSolver(SymbolicSolver):
    def __init__(self, neural):
        super().__init__()
        self.neural = neural
        self.neural_used = 0

    def solve_hybrid(self, board):
        cells = self.neural.predict_cell_priority(board)
        for r, c in cells:
            if board[r][c] != 0:
                continue
            values = self.neural.predict_values(board, r, c)
            self.neural_used += len(values)

            for num in values:
                if self.is_valid(board, r, c, num):
                    board[r][c] = num
                    if self.solve_hybrid(board):
                        return True
                    board[r][c] = 0
                    self.backtracks += 1
            return False
        return True

# ================= PUZZLE GENERATION =================
def generate_puzzle(removal=0.6):
    grid = np.zeros((9, 9), dtype=int)

    for i in range(0, 9, 3):
        nums = list(range(1, 10))
        random.shuffle(nums)
        grid[i:i+3, i:i+3] = np.array(nums).reshape(3, 3)

    temp = SymbolicSolver()
    temp.solve_pure(grid)

    puzzle = grid.copy()
    remove = int(81 * removal)
    cells = [(r, c) for r in range(9) for c in range(9)]
    random.shuffle(cells)

    for r, c in cells[:remove]:
        puzzle[r][c] = 0

    return puzzle

# ================= PRINT GRID =================
def print_grid(board, title):
    print("\n" + "=" * 30)
    print(title)
    print("=" * 30)
    for i in range(9):
        if i % 3 == 0 and i != 0:
            print("-" * 21)
        for j in range(9):
            if j % 3 == 0:
                print("|", end=" ")
            print(board[i][j] if board[i][j] != 0 else ".", end=" ")
        print("|")
    print("=" * 30)

# ================= MAIN RUN =================
def run_project():
    print("HYBRID NEURO-SYMBOLIC SUDOKU SOLVER")

    puzzle = generate_puzzle(0.65)
    print_grid(puzzle, "INPUT PUZZLE")

    # Pure symbolic
    pure_board = puzzle.copy()
    pure_solver = SymbolicSolver()
    t1 = time.time()
    pure_solver.solve_pure(pure_board)
    t1 = (time.time() - t1) * 1000

    # Hybrid
    hybrid_board = puzzle.copy()
    hybrid_solver = HybridSolver(NeuralGuide())
    t2 = time.time()
    hybrid_solver.solve_hybrid(hybrid_board)
    t2 = (time.time() - t2) * 1000

    print_grid(hybrid_board, "HYBRID SOLUTION")

    print("\nPERFORMANCE COMPARISON")
    print(f"Pure Backtracks   : {pure_solver.backtracks}")
    print(f"Hybrid Backtracks : {hybrid_solver.backtracks}")
    print(f"Pure Time (ms)    : {t1:.2f}")
    print(f"Hybrid Time (ms)  : {t2:.2f}")

# ================= RUN =================
run_project()


HYBRID NEURO-SYMBOLIC SUDOKU SOLVER

INPUT PUZZLE
| . 2 . | . 1 . | . . . |
| 1 . . | . . 2 | 3 . . |
| . 7 6 | 8 . . | 1 . 2 |
---------------------
| . 1 . | . 3 . | . 8 7 |
| 7 . . | 4 5 . | . . . |
| . . . | 1 . . | . . . |
---------------------
| 6 . 2 | 9 . . | 5 . . |
| . 5 . | . . 3 | 8 . . |
| 8 3 . | . . 4 | 6 . . |

HYBRID SOLUTION
| 5 2 4 | 3 1 9 | 7 6 8 |
| 1 9 8 | 7 6 2 | 3 4 5 |
| 3 7 6 | 8 4 5 | 1 9 2 |
---------------------
| 4 1 5 | 2 3 6 | 9 8 7 |
| 7 6 9 | 4 5 8 | 2 3 1 |
| 2 8 3 | 1 9 7 | 4 5 6 |
---------------------
| 6 4 2 | 9 8 1 | 5 7 3 |
| 9 5 1 | 6 7 3 | 8 2 4 |
| 8 3 7 | 5 2 4 | 6 1 9 |

PERFORMANCE COMPARISON
Pure Backtracks   : 3129
Hybrid Backtracks : 11
Pure Time (ms)    : 222.55
Hybrid Time (ms)  : 177.81
